In [0]:
%sql
create volume if not exists workspace.olist_landing.autoloader_metadata

In [0]:

from pyspark.sql.functions import current_timestamp, col, regexp_extract # regexp_extract -> get the batch from file path

catalog = "workspace"
schema = "olist_landing"

batch_path = f"/Volumes/{catalog}/{schema}/batched_files"
autoload_path = f"/Volumes/{catalog}/{schema}/autoloader_metadata"

def ingest_bronze (entity_name):
    source_path = f"{batch_path}/batch*/{entity_name}"
    schema_location = f"{autoload_path}/schemas/{entity_name}"
    checkpoint_location = f"{autoload_path}/checkpoints/{entity_name}"
    target_table = f"{catalog}.{schema}.bronze_{entity_name}"
  

    df = (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.schemaLocation", schema_location)
            .option("cloudFiles.inferColumnTypes", "true")
            .load(source_path)
            .withColumn("_souerce_file",col("_metadata.file_path"))
            .withColumn("_ingestion_ts", current_timestamp())
            .withColumn("_source_batch",regexp_extract(col("_metadata.file_path"),r"batch_(\d+)",1))
    )

    (  
        df.writeStream 
            .format("delta") 
            .option("checkpointLocation", checkpoint_location) 
            .trigger(availableNow=True)
            .toTable(target_table)
    )         



In [0]:
%sql
UPDATE  workspace.olist_landing.bronze_orders
set _source_batch = regexp_extract(_souerce_file, '(batch_[0-9]+)', 1)
where _source_batch is null or 
  _source_batch = ''

In [0]:

ingest_bronze("items")

In [0]:
%sql
select `_source_batch`, count(*)
from workspace.olist_landing.bronze_items
group by `_source_batch`

In [0]:
%sql
select *
from workspace.olist_landing.bronze_items
limit 100

In [0]:
ingest_bronze("orders")

In [0]:
%sql
select `_source_batch`, count(*)
from workspace.olist_landing.bronze_orders
group by `_source_batch`

In [0]:
ingest_bronze("payments")

In [0]:
%sql
select `_source_batch`, count(*)
from workspace.olist_landing.bronze_payments
group by `_source_batch`

In [0]:
## Reference Data Ingestion
from pyspark.sql.functions import current_timestamp, col

catalog = "workspace"
schema = "olist_landing"
data_path = f"/Volumes/{catalog}/{schema}/raw_files/"

def ingest_reference_data(entity_name, file_name=None):

    if file_name is None:
        file_name = f"olist_{entity_name}_dataset.csv"

    target_table = f"{catalog}.{schema}.bronze_{entity_name}"
    file_path = f"{data_path}{file_name}"

    df = (spark.read
                .format("csv")
                .option("header", "true")
                .option("inferSchema", "true" )
                .csv(file_path) 
                .withColumn("_souerce_file",col("_metadata.file_path"))
                .withColumn("_ingestion_ts", current_timestamp())
    )
        
    (  
            df.write 
                .format("delta") 
                .mode("overwrite") 
                .option("overwriteSchema", "true") 
                .saveAsTable(target_table)
    )         



In [0]:

ingest_reference_data("customers")


In [0]:
ingest_reference_data("geolocation")
ingest_reference_data("order_reviews")
ingest_reference_data("products")
ingest_reference_data("sellers")
ingest_reference_data("product_category_name_translation","product_category_name_translation.csv")


In [0]:
%sql
select * from workspace.olist_landing.bronze_product_category_name_translation